# Settled paper bets — money view

Re-run after every `python -m src.paper_trader --settle-only`.

**Edit `DATE` below** to set the per-day breakdown. `DATE` is the day the *match was played* (not when we settled it), so timezone of the script run doesn't matter. Set to `None` to skip the per-day section.

In [ ]:
DATE = '2026-05-05'   # ← edit each session. None to skip per-day section.

import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SETTLED = REPO / 'data' / 'paper_trades' / 'settled.csv'

df = pd.read_csv(SETTLED) if SETTLED.exists() else pd.DataFrame()
if not df.empty:
    df['event_date']    = pd.to_datetime(df['event_date']).dt.date.astype(str)
    df['settled_date']  = pd.to_datetime(df['timestamp_settled']).dt.date.astype(str)
    df['recorded_date'] = pd.to_datetime(df['timestamp_recorded']).dt.date.astype(str)
print(f'loaded {len(df)} settled bets')

In [ ]:
df.head()

## Headline

In [ ]:
def headline(df, date):
    if df.empty:
        print('No settled bets yet.'); return

    def _block(label, sub):
        if sub.empty:
            print(f'{label:14}  (no bets)'); return
        n = len(sub); wins = int(sub['bet_won'].sum())
        pnl = sub['net_pnl'].sum(); inv = sub['entry_price'].sum()
        roi = pnl / inv if inv > 0 else float('nan')
        print(f'{label:14}  bets={n:<4} wins={wins:<3} '
              f'win_rate={wins/n:.3f}  PnL=${pnl:+.2f}  '
              f'invested=${inv:.2f}  ROI={roi:+.2%}')

    print('=' * 82)
    if date is not None:
        _block(f'event={date}', df[df['event_date'] == date])
    _block('ALL-TIME', df)
    print('=' * 82)

headline(df, DATE)

## Equity curve & daily PnL

Grouped by event date (the day the match was played).

In [ ]:
def plot_equity(df):
    if df.empty:
        print('No settled bets.'); return
    daily = df.groupby('event_date')['net_pnl'].sum().sort_index()
    cum   = daily.cumsum()

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    ax1.plot(cum.index, cum.values, marker='o', lw=2, color='#2c3e50')
    ax1.fill_between(cum.index, cum.values, 0,
                     where=(cum.values >= 0), color='#2ecc71', alpha=0.2)
    ax1.fill_between(cum.index, cum.values, 0,
                     where=(cum.values < 0), color='#e74c3c', alpha=0.2)
    ax1.axhline(0, color='black', lw=0.5)
    ax1.set_ylabel('Cumulative PnL ($)')
    ax1.set_title(f'Equity curve — all-time PnL=${df["net_pnl"].sum():+.2f}  '
                  f'(n={len(df)} bets)')
    ax1.grid(alpha=0.3)

    colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in daily.values]
    ax2.bar(daily.index, daily.values, color=colors)
    ax2.axhline(0, color='black', lw=0.5)
    ax2.set_ylabel('Daily PnL ($)')
    ax2.set_xlabel('event date')
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

plot_equity(df)

## Per-bet detail (DATE only)

In [ ]:
def show_bets(df, date=None):
    sub = df if date is None else df[df['event_date'] == date]
    if sub.empty:
        print('(no rows for that date)'); return None
    cols = ['event_date', 'chosen_player_name', 'player_a', 'player_b',
            'chosen_direction', 'entry_price', 'theo_chosen',
            'bet_won', 'net_pnl', 'kalshi_url']
    out = (sub[cols]
             .sort_values('event_date', ascending=False)
             .reset_index(drop=True)
             .rename(columns={
                 'chosen_player_name': 'bet_on',
                 'chosen_direction':   'side',
                 'entry_price':        'cost',
                 'theo_chosen':        'theo',
             }))

    def color_row(row):
        bg = '#d5f5e3' if row['bet_won'] else '#fadbd8'
        return [f'background-color: {bg}'] * len(row)

    return (out.style
              .apply(color_row, axis=1)
              .format({'cost': '{:.2f}', 'theo': '{:.3f}', 'net_pnl': '{:+.3f}'}))

show_bets(df, DATE)

## Per-bet detail (all-time)

In [ ]:
show_bets(df, date=None)

## Feature attribution (signed, all-time)

Each feature's share of the log-odds shift toward each bet × that bet's realized PnL, summed across bets. Positive (green) = feature earns money when it drives bets. Negative (red) = noise or counter-signal.

In [ ]:
def plot_attribution(df):
    if df.empty or 'feature_shifts_json' not in df.columns:
        print('No attribution data.'); return
    rows = []
    for _, r in df.iterrows():
        try:
            shifts = json.loads(r['feature_shifts_json'])
        except Exception:
            continue
        total = sum(abs(v) for v in shifts.values())
        if total <= 0:
            continue
        pnl = float(r['net_pnl'])
        for f, s in shifts.items():
            rows.append({'feature': f, 'attributed': (s / total) * pnl})
    if not rows:
        print('No attribution data.'); return

    agg = (pd.DataFrame(rows)
             .groupby('feature')['attributed']
             .sum()
             .sort_values())
    colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in agg.values]

    fig, ax = plt.subplots(figsize=(8, max(4, 0.32 * len(agg))))
    ax.barh(agg.index, agg.values, color=colors)
    ax.axvline(0, color='black', lw=0.5)
    ax.set_xlabel('Total attributed PnL ($)')
    ax.set_title(f'Feature attribution — n={len(df)} settled bets')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_attribution(df)

## Drill into a specific match

Set `MATCH` below to any substring of a player's name (case-insensitive). Shows the per-feature signed shifts that drove that bet, plus whether it won. Set to `''` to skip. Multiple matches → all shown stacked.

In [ ]:
def show_match_attribution(df, query):
    if df.empty:
        print('No settled bets.'); return
    if query:
        sub = df[df['chosen_player_name'].str.contains(query, case=False, na=False)]
    else:
        sub = df.copy()
    if sub.empty:
        print(f'No settled bet matching {query!r}'); return

    n = len(sub)
    fig, axes = plt.subplots(n, 1, figsize=(8, 2.4 * n))
    if n == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, sub.iterrows()):
        try:
            shifts = json.loads(row['feature_shifts_json'])
        except Exception:
            ax.set_visible(False); continue
        items  = sorted(shifts.items(), key=lambda kv: -abs(kv[1]))
        names  = [k for k, _ in items][::-1]
        vals   = [v for _, v in items][::-1]
        colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in vals]
        ax.barh(names, vals, color=colors)
        ax.axvline(0, color='black', lw=0.5)

        opponent = (row['player_b'] if row['chosen_player_name'] == row['player_a']
                    else row['player_a'])
        won = '✓ WON' if row['bet_won'] else '✗ LOST'
        ax.set_title(
            f"{row['chosen_player_name']} ({row['chosen_direction']}) vs {opponent}  "
            f"— {won}  pnl=${row['net_pnl']:+.3f}\n"
            f"theo={row['theo_chosen']:.3f}  cost={row['entry_price']:.2f}  "
            f"event_date={row['event_date']}",
            fontsize=9, loc='left')
        ax.tick_params(labelsize=8)
        ax.grid(axis='x', alpha=0.25)
    plt.tight_layout()
    plt.show()

# Set MATCH to a substring of any chosen_player_name (e.g. 'Castelnuovo',
# 'Clarke', 'Zhang'). '' = skip this section. Multiple matches → all shown.
MATCH = 'Santillan'
show_match_attribution(df, MATCH)